# 01 - Feature Extraction

Computes geometric features (jakteristics) for every file declared for a dataset in `configs/datasets.yaml`, and writes one `features_{file}.csv` per file to `ProcessedData/{dataset}/`

In [ ]:
import sys
from pathlib import Path

#sys.path.insert(0, str(Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()))
current_notebook_dir = Path.cwd()

if current_notebook_dir.name == "notebooks":
    project_root = current_notebook_dir.parent
else:
    project_root = current_notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import set_global_seed, load_config, get_logger
from data.paths import get_dataset_config, get_processed_file
from src.io_utils import load_raw_point_file, save_processed_features
from src.features import extract_features

logger = get_logger("01_feature_extraction")

In [ ]:
DATASET = "ahn3" #"paris_lille"

config = load_config("default")
set_global_seed(config["experiment"]["seed"])

dataset_cfg = get_dataset_config(DATASET)
files = dataset_cfg["files"]
radius = dataset_cfg.get("feature_radius")

print(f"Dataset: {DATASET}")
print(f"Files to process ({len(files)}): {files}")
print(f"Radius: {radius or '(using configs/datasets.yaml -> feature_extraction.default_radius)'}")

## Run feature extraction for every file

Each file is processed independently (raw parse -> feature compute -> save). NaN points (insufficient neighbors within radiu) and duplicate (x,y,z) points are reported per file, not silently dropped

In [ ]:
extraction_summary = []

for file_name in files:
    logger.info(f"--- {DATASET} / {file_name} ---")

    raw_df = load_raw_point_file(DATASET, file_name)
    feats_df = extract_features(raw_df, dataset=DATASET, radius=radius)

    out_path = save_processed_features(feats_df, DATASET, file_name)

    extraction_summary.append({
        "file": file_name,
        "n_points": len(feats_df),
        "n_nan_flagged": int(feats_df["__nan_feature__"].sum()),
        "pct_nan_flagged": round(100 * feats_df["__nan_feature__"].sum() / len(feats_df), 2),
        "output_path": str(out_path),
    })

extraction_summary

## Summary

- A high `pct_nan_flagged` for a particular file may indicate the radius isn't well matched to that file's point density. So we'll have to tweak tghis accordingly

In [ ]:
import pandas as pd
pd.DataFrame(extraction_summary)